# ML Testing & MLOps: A CI-Style Test Suite for a Diagnostic Classifier

**Problem statement.** You're the QA engineer embedded with an ML team that trains a tumor-malignancy
classifier on the real **Breast Cancer Wisconsin (Diagnostic)** dataset (569 real patient samples, 30
real cell-nucleus measurements, from `sklearn.datasets.load_breast_cancer`). Before this model is
allowed into a CI/CD pipeline, your job is to write the automated checks that gate the release — the
same three categories of test a real MLOps pipeline runs on every commit:

1. **Data validation / contract tests** — does incoming data match the schema the model was trained on?
2. **Model performance tests with a realistic failure mode** — a model can clear an overall-accuracy bar
   while silently failing on the class that matters most (missed malignant tumors). Your test suite must
   catch that, not just check accuracy.
3. **Data drift tests** — a statistical test that flags when a new batch of data no longer looks like the
   training distribution, run once on data that *shouldn't* trigger it and once on data that *should*, to
   prove the detector actually discriminates signal from noise (not just always-pass or always-fail).

**What you should produce**, as plain Python functions (assert-based, pytest-style but runnable directly
in this notebook — no pytest required):

- `validate_schema(df)` — raises an `AssertionError` with a specific, descriptive message if any contract
  is violated: expected columns present, all columns numeric, no nulls, and every feature within its
  observed physical range (e.g. a `mean radius` of -5 or 500 should fail, not just silently pass).
- `evaluate_model_thresholds(y_true, y_pred, min_accuracy=0.90, min_recall_malignant=0.90)` — returns a
  dict of named checks (`accuracy`, `recall_malignant`) each with `pass`/`fail` and the measured value.
  You must demonstrate a concrete case (by training a deliberately biased model) where the accuracy check
  passes but the recall check fails.
- `detect_drift(reference_df, new_df, alpha=0.05)` — runs a two-sample Kolmogorov–Smirnov test per
  feature and returns the list of feature names whose distribution shifted significantly (p < alpha). Run
  it on (a) two honest random splits of the same data (expect ~no drift) and (b) a copy of the data with
  one feature deliberately shifted (expect that feature, and only that feature, to be flagged).
- `run_test_suite(...)` — calls all of the above and prints a short CI-style PASS/FAIL report.

Work through the starter cells, then compare against the solution at the bottom.

## Setup

Imports and loading the real dataset.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score

RANDOM_STATE = 42


In [ ]:
# Load the real Breast Cancer Wisconsin (Diagnostic) dataset.
# target: 0 = malignant, 1 = benign (sklearn's convention for this dataset)
data = load_breast_cancer(as_frame=True)
df = data.frame.copy()
feature_cols = list(data.feature_names)

print(df.shape)
df.head()


## Part 1 — Data validation / contract tests

Write `validate_schema(df)` below. It should raise `AssertionError` (with a message that says *what*
failed and *why*) if:

- any expected column in `feature_cols` + `['target']` is missing
- any feature column is not numeric
- any column contains nulls
- any feature value falls outside a reasonable physical range — use
  `[reference_df[col].min() * 0.5, reference_df[col].max() * 1.5]` as the allowed band per column
  (a generous margin around what training data actually looked like)

Test it against the clean `df` (should pass silently) and against a corrupted copy you construct
yourself (should raise).

In [ ]:
def validate_schema(check_df, reference_df=df, expected_cols=feature_cols + ["target"]):
    """Raise AssertionError with a descriptive message on any contract violation."""
    # TODO: check all expected columns are present
    # TODO: check every feature column has a numeric dtype
    # TODO: check there are no nulls anywhere
    # TODO: check every feature value is within [0.5*min, 1.5*max] of reference_df for that column
    raise NotImplementedError


# TODO: call validate_schema(df) — should pass with no error

# TODO: build a corrupted copy of df (e.g. set one feature's values to -999, or drop a column,
# or inject a NaN) and confirm validate_schema raises AssertionError on it


## Part 2 — Model performance tests (and a realistic failure mode)

Train **two** models on the same train/test split:

1. A normal `LogisticRegression` on the real features.
2. A deliberately biased model that always predicts the majority class (`benign`) — simulating a model
   that overfit to class imbalance, or a regression in a retrain that silently collapsed to the prior.

Write `evaluate_model_thresholds(y_true, y_pred, min_accuracy=0.90, min_recall_malignant=0.90)`. Recall
for "malignant" matters more than overall accuracy here — a missed malignant tumor (false negative) is
the costly error, and accuracy alone can hide that. Confirm that the biased "always benign" model clears
the accuracy bar (because benign is the majority class) while *failing* the malignant-recall bar.

In [ ]:
X = df[feature_cols]
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)

# TODO: train a LogisticRegression (increase max_iter if it doesn't converge) on X_train/y_train
# model = ...

# TODO: build "biased_preds": an array the same length as y_test, always predicting the majority
# class label in y_train (look it up, don't hardcode which int it is)


def evaluate_model_thresholds(y_true, y_pred, min_accuracy=0.90, min_recall_malignant=0.90):
    """Return {'accuracy': {...}, 'recall_malignant': {...}} each with 'value' and 'pass'."""
    # TODO: compute accuracy_score and recall_score (malignant is label 0 in this dataset —
    # pass pos_label=0 to recall_score)
    # TODO: build and return the result dict, each entry with keys 'value' and 'pass'
    raise NotImplementedError


# TODO: run evaluate_model_thresholds on the real model's predictions, and again on biased_preds.
# Print both and confirm the biased model passes accuracy but fails recall_malignant.


## Part 3 — Data drift test

Write `detect_drift(reference_df, new_df, alpha=0.05)`: for every column in `feature_cols`, run
`scipy.stats.ks_2samp(reference_df[col], new_df[col])` and collect the names of columns whose p-value is
below `alpha`.

Then prove the detector actually works, not just that it runs:

- **Case A (no real drift):** split `df` into two random halves and run `detect_drift` between them.
  Expect the flagged list to be small/empty (occasional false positives at `alpha=0.05` are expected and
  fine — that's the point of picking a stricter alpha in practice).
- **Case B (injected drift):** copy one half, and deliberately shift a single column (e.g. add
  `5 * reference_df[col].std()` to every value of `"mean radius"` in the copy — a synthetic, controlled
  shift, which is the right use of synthetic data here since we need to *know* which feature drifted to
  check the detector finds exactly that one). Confirm `detect_drift` flags that column (and ideally not
  many others).

In [ ]:
def detect_drift(reference_df, new_df, alpha=0.05, columns=feature_cols):
    """Return the list of column names where the KS-test p-value < alpha."""
    # TODO: loop over `columns`, run stats.ks_2samp(reference_df[col], new_df[col]) on each,
    # collect names where pvalue < alpha
    raise NotImplementedError


# TODO: Case A — split df into two random halves (e.g. via train_test_split with test_size=0.5),
# run detect_drift between them, print the flagged columns

# TODO: Case B — copy one half, shift "mean radius" by 5 * std, run detect_drift against the
# unshifted half, confirm "mean radius" is in the flagged list


## Part 4 — Put it together: a CI-style report

Write `run_test_suite()` that calls all three checks above on the model and data defined earlier, and
prints a short report, e.g.:

```
[PASS] schema_validation
[PASS] accuracy >= 0.90 (0.97)
[FAIL] recall_malignant >= 0.90 (0.83)   <- only if you run it on the biased model
[PASS] drift_check (0 features flagged)
```


In [ ]:
def run_test_suite(model_preds, y_true, ref_df, new_df):
    # TODO: call validate_schema, evaluate_model_thresholds, and detect_drift,
    # then print a PASS/FAIL line per check
    raise NotImplementedError

# TODO: run it once against the real LogisticRegression's predictions


---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
# --- Part 1: schema / contract validation ---
def validate_schema(check_df, reference_df=df, expected_cols=feature_cols + ["target"]):
    missing = set(expected_cols) - set(check_df.columns)
    assert not missing, f"Missing expected columns: {missing}"

    for col in feature_cols:
        assert pd.api.types.is_numeric_dtype(check_df[col]), f"Column '{col}' is not numeric"

    nulls = check_df[expected_cols].isnull().sum()
    bad_null_cols = nulls[nulls > 0]
    assert bad_null_cols.empty, f"Null values found in columns: {dict(bad_null_cols)}"

    for col in feature_cols:
        lo, hi = reference_df[col].min() * 0.5, reference_df[col].max() * 1.5
        out_of_range = check_df[(check_df[col] < lo) | (check_df[col] > hi)]
        assert out_of_range.empty, (
            f"Column '{col}' has {len(out_of_range)} values outside expected range "
            f"[{lo:.2f}, {hi:.2f}]"
        )

validate_schema(df)  # passes silently

corrupt = df.copy()
corrupt.loc[0, "mean radius"] = -999
try:
    validate_schema(corrupt)
    print("FAILED TO CATCH CORRUPTION")
except AssertionError as e:
    print("Caught corruption as expected:", e)


# --- Part 2: model performance thresholds, including the accuracy-hides-recall failure mode ---
model = LogisticRegression(max_iter=10000, random_state=RANDOM_STATE)
model.fit(X_train, y_train)
real_preds = model.predict(X_test)

majority_label = y_train.value_counts().idxmax()
biased_preds = np.full_like(y_test, majority_label)

def evaluate_model_thresholds(y_true, y_pred, min_accuracy=0.90, min_recall_malignant=0.90):
    acc = accuracy_score(y_true, y_pred)
    rec_malignant = recall_score(y_true, y_pred, pos_label=0)
    return {
        "accuracy": {"value": acc, "pass": acc >= min_accuracy},
        "recall_malignant": {"value": rec_malignant, "pass": rec_malignant >= min_recall_malignant},
    }

print("Real model:", evaluate_model_thresholds(y_test, real_preds))
print("Biased 'always benign' model:", evaluate_model_thresholds(y_test, biased_preds))
# -> the biased model clears the accuracy bar (benign is the majority class) but
#    fails recall_malignant (0.0 — it never predicts malignant at all), exactly
#    the silent-failure mode an accuracy-only gate would miss.


# --- Part 3: drift detection, validated on a known shift ---
def detect_drift(reference_df, new_df, alpha=0.05, columns=feature_cols):
    flagged = []
    for col in columns:
        _, pvalue = stats.ks_2samp(reference_df[col], new_df[col])
        if pvalue < alpha:
            flagged.append(col)
    return flagged

half_a, half_b = train_test_split(df, test_size=0.5, random_state=RANDOM_STATE)
print("Case A (no real drift) flagged:", detect_drift(half_a, half_b))

shifted = half_b.copy()
shifted["mean radius"] = shifted["mean radius"] + 5 * df["mean radius"].std()
print("Case B (injected drift) flagged:", detect_drift(half_a, shifted))
# -> Case A should flag few/no columns; Case B should flag 'mean radius'


# --- Part 4: CI-style report ---
def run_test_suite(model_preds, y_true, ref_df, new_df):
    results = []

    try:
        validate_schema(ref_df)
        results.append(("schema_validation", True, None))
    except AssertionError as e:
        results.append(("schema_validation", False, str(e)))

    perf = evaluate_model_thresholds(y_true, model_preds)
    for name, outcome in perf.items():
        results.append((name, outcome["pass"], f"value={outcome['value']:.3f}"))

    drifted = detect_drift(ref_df, new_df)
    results.append(("drift_check", len(drifted) == 0, f"{len(drifted)} features flagged: {drifted}"))

    for name, passed, detail in results:
        status = "PASS" if passed else "FAIL"
        print(f"[{status}] {name}" + (f" ({detail})" if detail else ""))

run_test_suite(real_preds, y_test, half_a, half_b)
```

**Why this matters for ML quality engineering:** every one of these checks maps directly onto a real
CI/CD gate for models — schema/contract tests catch upstream pipeline bugs before they reach the model,
threshold tests on the *right* metric (not just accuracy) catch models that are technically "fine" but
operationally dangerous, and drift tests validated against a known injected shift (rather than trusted
blindly) are what separates a monitoring check that actually works from one that just always prints
green.

</details>